In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import gng_py
import time

import json
import numpy as np
import pandas as pd
import math

# Helper Functions

In [ ]:

# Result processing
#- Produces a DataFrame with:
#    - neruon id 
#    - weights/position 
#    - hits per class
#- Requires:
#    - model_string from GNG
#    - targets for class hit amount

def processGngModel(model_string,y_train):
    data = json.loads(model_string)
    points = None
    edges = None
    edge_positions = None
    rows = []
    for neuron in data["model"]["neurons"]:
        position = neuron["position"]
        id = neuron['id']
        x = position[0]
        y = position[1]
        rows.append({"id": id, "x": x, "y": y})

    points = pd.DataFrame(rows)

    num_classes = len(np.unique(y_train))


    for a in data["model"]["neurons"]:
        a["hits"] = np.array(np.zeros(num_classes))

    df = pd.DataFrame(data["model"]["neurons"])
    return df

# Calculate Hits
#  - Returns the cass hits, every neurons amount of being BMU
def getHits(X,y,df):

    num_samples = len(X)
    num_weights = len(df.at[0,"position"])
    num_neurons = len(df)
    input_width = X.shape[1]
    num_classes = len(np.unique(y)) 


    # For all samples
    for s in range(0,num_samples):
        # get sample position
        sample_pos = X[s]
        # init neuron dist arr
        dist = np.zeros(num_neurons)

        # For all Neurons
        for i,row in df.iterrows():
            # get neuron position
            neuron_pos = row["position"]
            # init distance val
            dist_val = 0.0

            # For all neuron weights (all dimensions)
            for a in range(0,num_classes):
                # add squared distances
                diff = sample_pos[a]-neuron_pos[a]
                dist_val += diff * diff

            dist[i] = np.sqrt(dist_val)

        # find best neuron (winner)
        best_neuron_idx = np.argmin(dist)
        sample_class = y[s]

        hits = df.at[best_neuron_idx,"hits"]
        hits[sample_class] +=1

    return df

    
def getHits(X,y,df):

    num_samples = len(X)
    num_weights = len(df.at[0,"position"])
    num_neurons = len(df)
    input_width = X.shape[1]
    num_classes = len(np.unique(y)) 


    # For all samples
    for s in range(0,num_samples):
        # get sample position
        sample_pos = X[s]
        # init neuron dist arr
        dist = np.zeros(num_neurons)

        # For all Neurons
        for i,row in df.iterrows():
            # get neuron position
            neuron_pos = row["position"]
            # init distance val
            dist_val = 0.0

            # For all neuron weights (all dimensions)
            for a in range(0,num_classes):
                # add squared distances
                diff = sample_pos[a]-neuron_pos[a]
                dist_val += diff * diff

            dist[i] = np.sqrt(dist_val)

        # find best neuron (winner)
        best_neuron_idx = np.argmin(dist)
        sample_class = y[s]

        hits = df.at[best_neuron_idx,"hits"]
        hits[sample_class] +=1

    return df

def detect(best_neurons,y,df):
    num_classes = len(np.unique(y)) 
    # -------------------------------------------------
    # Assign class labels to samples using top-k neurons
    # -------------------------------------------------
    num_samples = len(y)
    k = 3  # Use top 3 nearest neurons
    sample_predictions = []

    for s in range(num_samples):
        # Get top k neurons for this sample
        top_k_neurons = best_neurons[best_neurons["sample_idx"] == s].head(k)

        # Initialize class probabilities
        class_probs = np.zeros(num_classes)

        # For each of the top k neurons
        for idx, row in top_k_neurons.iterrows():
            neuron_id = row["neuron_id"]
            rank = row["rank"]

            # Get neuron hits
            neuron_row = df[df["id"] == neuron_id]
            hits = np.array(neuron_row.iloc[0]["hits"])
            total_hits = np.sum(hits)

            # Calculate weight based on rank (lower rank = higher weight)
            weight = 1.0 / (rank + 1)

            # Add weighted class probabilities
            if total_hits > 0:
                class_probs += (hits / total_hits) * weight
            else:
                # If no hits, distribute uniformly
                class_probs += (np.ones(num_classes) / num_classes) * weight

        # Normalize probabilities
        class_probs = class_probs / np.sum(class_probs)

        # Predict class with highest probability
        predicted_class = np.argmax(class_probs)

        sample_predictions.append({
            "sample_idx": s,
            "predicted_class": predicted_class,
            "class_probs": class_probs,
            "true_class": y[s]
        })


    predictions_df = pd.DataFrame(sample_predictions)

    return predictions_df



def findBMU(X_test,gng_res):
    # -------------------------------------------------
    # Inference: Find best matching neurons for X_test
    # -------------------------------------------------
    num_samples_test = len(X_test)
    num_neurons = len(gng_res)
  #  num_classes = len(np.unique(y_train))
    input_width = X_test.shape[1]
    
    # Store results: list of tuples (sannmple_idx, neuron_idx, distance, neuron_id)
    best_neurons = []
    
    # For all test samples
    for s in range(num_samples_test):
        # get sample position
        sample_pos = X_test[s]
        # init neuron dist arr
        dist = np.zeros(num_neurons)
    
        # For all Neurons
        for i, row in gng_res.iterrows():
            # get neuron position
            neuron_pos = row["position"]
           # print(neuron_pos)
            # init distance val
            dist_val = 0.0
    
            # For all neuron weights (all dimensions)
            for a in range(input_width):
                # add squared distances
                diff = sample_pos[a] - neuron_pos[a]
                dist_val += diff * diff
    
            # neuron dist val arr[curr neuron] = sqrt of dist val
            dist[i] = np.sqrt(dist_val)
    
        # Sort neurons by distance (smallest fi
    
    #    print(dist)
        sorted_indices = np.argsort(dist)
    #    print(sorted_indices)
        
        # Store all neurons for this sample, sorted by distance
        for rank, neuron_idx in enumerate(sorted_indices):
            neuron_id = gng_res.at[neuron_idx, "id"]
            distance = dist[neuron_idx]
            best_neurons.append({
                "sample_idx": s,
                "rank": rank,
                "neuron_id": neuron_id,
                "distance": distance
            })
    
    # Convert to DataFrame for easy viewing
    best_neurons_df = pd.DataFrame(best_neurons)
    return best_neurons_df

# Data preparation

In [3]:
# -------------------------------------------------
# Load & preprocess data
# -------------------------------------------------
iris = load_iris()
X = iris.data
y = iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# GNG calculation

In [9]:
ctx = gng_py.PyContext()
ctx.create_system()

ctx.set_parameters(
            input_width = 4,
            weight_rng_min = -1.1,
            weight_rng_max = 1.1,
            edge_removal_age = 50,
            neuron_creation_interval = 200,
            max_train_iterations = 20000,
            target_error = 0.096,
            epsilon_w = 0.1,
            epsilon_n = 0.006,
            alpha = 0.5,
            beta = 0.995,
)

ctx.init_dataset_vec(X_train.flatten())
start = time.time()
ctx.fit()
end = time.time()
model_string = ctx.get_model_string()
print("Time for gng calculation: ",end-start)

Time for gng calculation:  0.42705750465393066


In [10]:
y_train

array([0, 2, 1, 0, 1, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 0, 0, 2, 2, 0, 1, 0,
       2, 0, 1, 2, 2, 0, 2, 0, 0, 1, 1, 0, 2, 2, 1, 1, 2, 1, 0, 1, 0, 2,
       0, 0, 2, 0, 0, 0, 0, 1, 2, 1, 0, 2, 1, 2, 0, 2, 0, 1, 2, 0, 1, 1,
       2, 1, 1, 2, 0, 0, 0, 2, 1, 2, 1, 2, 2, 1, 0, 2, 1, 0, 2, 0, 2, 1,
       1, 0, 1, 2, 0, 0, 2, 2, 2, 1, 2, 0, 2, 1, 2, 2, 0, 1, 1, 1, 1, 1,
       0, 2, 1, 1, 0, 0, 0, 0, 1, 0])

In [5]:
df = processGngModel(model_string,y_train)

In [6]:
df = getHits(X_train,y_train,df)

In [7]:

best_neurons_test = findBMU(X_test,df)
best_neurons_train = findBMU(X_train,df)

pred_train = detect(best_neurons_train,y_train,df)
pred_test = detect(best_neurons_test,y_test,df)

In [8]:

# Calculate accuracy
accuracy_train = np.mean(pred_train["predicted_class"] == pred_train["true_class"])
accuracy_test = np.mean(pred_test["predicted_class"] == pred_test["true_class"])
print(f"\nAccuracy Train using GNG: {accuracy_train:.4f}")
print(f"\nAccuracy Test  using GNG: {accuracy_test:.4f}")


Accuracy Train using GNG: 0.9917

Accuracy Test  using GNG: 0.9667
